In [ ]:
! pip install oauth2client
! pip install gspread
! pip install gspread_dataframe
from google.oauth2.service_account import Credentials
import gspread
import pandas as pd
from gspread_dataframe import get_as_dataframe
from collections import defaultdict
import re
from datetime import datetime
import decimal
from pyspark.sql import functions as F
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from gspread.exceptions import APIError
import os, json

In [ ]:
# Credentials
info = json.loads(os.environ["GOOGLE_SERVICE_ACCOUNT_JSON"])
scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive.file",
]
creds = Credentials.from_service_account_info(info, scopes=scopes)
client = gspread.authorize(creds)

In [ ]:
orders_spreadsheet = client.open("Lakehouse Orders")
attendance_spreadsheet = client.open("Lakehouse Attendance")
earnings_spreadsheet = client.open("Lakehouse Earnings")
loginhours_spreadsheet = client.open("Lakehouse Login Hours")
store_map = client.open("Store Map")
rider_type = client.open("Rider Type")

In [ ]:
# Clean column names and drop empty rows
def clean_column_names_drop_empty(df):
    # Drop columns with empty or NaN names
    df = df.loc[:, df.columns.notnull()]
    df = df.loc[:, df.columns != ""]
    df = df.loc[:, df.columns.str.strip() != ""]

    # Clean all column names
    clean_cols = []
    for col in df.columns:
        new_col = col.strip()
        new_col = re.sub(r"[ ,;{}\(\)\n\t=\\/:*?\"<>|]", "_", new_col)
        new_col = re.sub(r"_+", "_", new_col)  # collapse multiple underscores
        new_col = new_col.strip("_")  # remove leading/trailing _
        clean_cols.append(new_col)

    df.columns = clean_cols
    return df

def get_ordinal_suffix(n):
    if 10 <= n % 100 <= 20:
        return "th"
    else:
        return {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")

day_columns = [f"{i}{get_ordinal_suffix(i)}" for i in range(1, 32)]

month_map = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
}
current_year = datetime.now().year

In [ ]:
# Retry-safe fetch for a single worksheet
def fetch_worksheet(sheet, retries=7, delay=3):
    """
    Fetch one worksheet with retries to handle 503/429 errors.
    """
    for attempt in range(retries):
        try:
            values = sheet.get_all_values()
            if not values:
                return None
            df = pd.DataFrame(values[1:], columns=values[0])
            df = clean_column_names_drop_empty(df)
            return sheet.title, df
        except Exception as e:
            print(f"Error fetching {sheet.title}, attempt {attempt+1}/{retries}: {e}")
            time.sleep(delay * (2**attempt))  # backoff
    return None


In [ ]:
# Load all worksheets in parallel
def load_sheets(spreadsheet, max_workers=5):
    """
    Fetch all sheets in a workbook in parallel, return {sheet_name: df}.
    """
    dfs = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_worksheet, sheet): sheet for sheet in spreadsheet.worksheets()}
        for future in as_completed(futures):
            result = future.result()
            if result:
                sheet_name, df = result
                dfs[sheet_name] = df
    return dfs

In [ ]:
# Parallel group_by_client
def group_by_client(dfs, max_workers=5):
    """
    Group sheets into client DataFrames in parallel.
    """
    client_dfs = defaultdict(list)

    def process_sheet(sheet_name, df):
        parts = sheet_name.strip().split("_")
        if len(parts) != 3:
            print("Skipping (bad format) : ", sheet_name)
            return None
        month, client, value = parts
        df = df.copy()
        df["Month"] = month
        return client, df

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_sheet, name, df): name for name, df in dfs.items()}
        for future in as_completed(futures):
            result = future.result()
            if result:
                client, df = result
                client_dfs[client].append(df)

    return {client: pd.concat(df_list, ignore_index=True) for client, df_list in client_dfs.items()}

In [ ]:
def process_category(category_dfs, value_column_name, table_name):
    """
    Process each clients together and save to Spark tables.

    Args:
        category_dfs: dict of {client: dataframe}
        value_column_name: str, name for the unpivoted value column ("Orders", "Attendance", "Earnings")
        table_name: str, name of the final table saved
    """
    final_frames = []

    # Combine per client and unpivot
    for client, df in category_dfs.items():

        # Keep only valid day columns
        valid_day_cols = [col for col in day_columns if col in df.columns]
        if not valid_day_cols:
            print(f"No day columns found for {client} in {table_name}, skipping.")
            continue

        safe_value_name = f"__{value_column_name}__"

        # Drop stray metric columns (keep only the one we’re processing)
        metric_columns = {"Orders", "Earnings", "LoginHours", "Attendance"}
        stray_metrics = [c for c in df.columns if c in metric_columns and c != value_column_name]
        df = df.drop(columns=stray_metrics, errors="ignore")

        df_unpivoted = pd.melt(
            df,
            id_vars = [col for col in df.columns if col not in valid_day_cols],
            value_vars = valid_day_cols,
            var_name = "Day_Ordinal",
            value_name = safe_value_name
        )

        # Standardise back to the original column name
        df_unpivoted.rename(columns = {safe_value_name: value_column_name}, inplace = True)

        dupes = [c for c in df_unpivoted.columns if c == value_column_name]
        if len(dupes) > 1:
            df_unpivoted = df_unpivoted.loc[:, ~df_unpivoted.columns.duplicated(keep = "last")]

        # Add client column
        df_unpivoted["Client"] = client

        # Convert Day_Ordinal to number
        df_unpivoted["Day_Num"] = df_unpivoted["Day_Ordinal"].str.extract(r"(\d+)").astype(int)

        # Month number
        df_unpivoted["Month_Num"] = df_unpivoted["Month"].map(month_map)
                
        # Proper date
        df_unpivoted["Date"] = pd.to_datetime({
            "year": current_year,
            "month": df_unpivoted["Month_Num"],
            "day": df_unpivoted["Day_Num"]
        }, errors='coerce')

        # Drop invalid dates
        df_unpivoted = df_unpivoted[df_unpivoted["Date"].notna()].copy()

        # Convert types
        if value_column_name == "Orders":
            df_unpivoted[value_column_name] = (
                pd.to_numeric(df_unpivoted[value_column_name], errors="coerce")
                .fillna(0)
                .astype(int)
            )

        elif value_column_name in ["Earnings", "LoginHours"]:
            df_unpivoted[value_column_name] = (
                pd.to_numeric(df_unpivoted[value_column_name], errors="coerce")
                .fillna(0.0)  
                .astype(float) 
            )

        for date_col in ["Date_of_Deployment", "Date_of_Return"]:
            if date_col in df_unpivoted.columns:
                df_unpivoted[date_col] = pd.to_datetime(df_unpivoted[date_col], errors="coerce").dt.date

        # Cleanup helper/day columns
        df_unpivoted.drop(columns=["Day_Num", "Month_Num", "Day_Ordinal"], inplace=True, errors="ignore")
        df_unpivoted.drop(columns=[col for col in df.columns if col in day_columns], inplace=True, errors="ignore")

        # Remove dupliate rows
        df_unpivoted = df_unpivoted.drop_duplicates().reset_index(drop = True)

        final_frames.append(df_unpivoted)

    if not final_frames:
        print(f"No data to save for {table_name}.")
        return pd.DataFrame()

    final_df = pd.concat(final_frames, ignore_index = True).drop_duplicates().reset_index(drop = True)

    # Save to Spark
    spark_df = spark.createDataFrame(final_df).withColumn("Date", F.col("Date").cast("date"))
    spark_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(f"Tables/{table_name}")

    spark.sql(f"VACUUM delta.`Tables/{table_name}` RETAIN 168 HOURS")

    print(f"Saved + Vacuumed table: {table_name}")
    return final_df


In [ ]:
# Load orders data
orders_dfs = group_by_client(load_sheets(orders_spreadsheet))

In [ ]:
# Load attendance data
attendance_dfs = group_by_client(load_sheets(attendance_spreadsheet))

In [ ]:
# Load earnings data
earnings_dfs = group_by_client(load_sheets(earnings_spreadsheet))

In [ ]:
# Load login hours data
loginhours_dfs = group_by_client(load_sheets(loginhours_spreadsheet))

In [ ]:
# Process data in all workbooks and push to Spark tables
orders_df = process_category(orders_dfs, "Orders", "Orders_Master")
attendance_df = process_category(attendance_dfs, "Attendance", "Attendance_Master")
earnings_df = process_category(earnings_dfs, "Earnings", "Earnings_Master")
loginhours_df = process_category(loginhours_dfs, "LoginHours", "Login_Hours_Master")

In [ ]:
# Create Calendar table
start_date = pd.to_datetime(f"{current_year}-01-01")
end_date = pd.to_datetime(f"{current_year}-12-31")

calendar_df = pd.DataFrame({"Date": pd.date_range(start=start_date, end=end_date, freq="D")})

# Year
calendar_df["Year"] = calendar_df["Date"].dt.year

# Quarter (text + numeric for sorting)
calendar_df["Quarter_Num"] = calendar_df["Date"].dt.quarter
calendar_df["Quarter"] = "Quarter " + calendar_df["Quarter_Num"].astype(str)

# Month (text + numeric for sorting)
calendar_df["Month_Num"] = calendar_df["Date"].dt.month
calendar_df["Month"] = calendar_df["Date"].dt.strftime("%B")  # January, February, etc.

# Week (ISO week number, 1–52/53 depending on year)
calendar_df["Week_Num"] = calendar_df["Date"].dt.isocalendar().week

# Year-Month (for visuals & slicers)
calendar_df["YearMonth"] = calendar_df["Date"].dt.strftime("%Y-%b")  # e.g., 2025-Jan
calendar_df["YearMonth_Num"] = calendar_df["Year"] * 100 + calendar_df["Month_Num"]  # for sorting

# Store Date as just date (no time)
calendar_df["Date"] = calendar_df["Date"].dt.date

# Save to Spark
spark_calendar_df = spark.createDataFrame(calendar_df)
spark_calendar_df.write.mode("overwrite").format("delta").save("Tables/Calendar")

print("Saved table: Calendar")

In [ ]:
# Create master ID table

unique_rows = []

# Merge all client DataFrames into one big dictionary
combined_client_dfs = {**orders_dfs, **attendance_dfs, **earnings_dfs}

# Build Unique ID + Name master
for client, df in combined_client_dfs.items():
    if "ID" in df.columns:
        df = df.copy()
        df["ID"] = df["ID"].astype(str).str.strip().str.upper()
        cols = ["ID"]
        if "Phone_Number" in df.columns:
            cols.append("Phone_Number")

        subset = df[cols].drop_duplicates().copy()

        # Replace missing IDs with Phone_Number
        if "Phone_Number" in subset.columns:
            subset.loc[subset["ID"] == "-", "ID"] = subset.loc[subset["ID"] == "-", "Phone_Number"]

        unique_rows.append(subset)

if unique_rows:
    combined_unique_df = pd.concat(unique_rows, ignore_index = True)

    combined_unique_df = combined_unique_df.sort_values("ID").drop_duplicates(subset = ["ID"], keep = "first").reset_index(drop = True)

    # Keeping only relevant columns
    final_unique_df = combined_unique_df[["ID"] + (["Phone_Number"] if "Phone_Number" in combined_unique_df.columns else [])]

else:
    final_unique_df = pd.DataFrame(columns=["ID", "Phone_Number"])

# Save to Spark
spark_unique_df = spark.createDataFrame(final_unique_df)
spark_unique_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save("Tables/Unique_ID_Master")
print("Saved table: Unique_ID_Master")

In [ ]:
# Create the ABC Classification disconnected table
ABC_Classification = pd.DataFrame({
    "Classification": ["Gold", "Silver", "Bronze"]
})

# Convert to Spark
spark_df = spark.createDataFrame(ABC_Classification)

# Save into Lakehouse
spark_df.write.mode("overwrite").format("delta").save("Tables/ABC_Classification")
print("Saved table: ABC_Classification")

In [ ]:
# Create a table of all clients
def client_list_master(orders_df, earnings_df, attendance_df):
    # Collect all clients
    all_clients = pd.concat([
        orders_df[["Client"]],
        earnings_df[["Client"]],
        attendance_df[["Client"]]
    ], ignore_index=True).drop_duplicates().reset_index(drop=True)

    spark_df = spark.createDataFrame(all_clients)
    spark_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save("Tables/Client_List_Master")

    print("Saved table: Client_List_Master")
    return all_clients

In [ ]:
# Executing Client List table
client_list_master(orders_df, earnings_df, attendance_df)

In [ ]:
# Adding a store mapping google sheet for riders
sheet = store_map.worksheet("Stores Names Client 1")

title, store_df = fetch_worksheet(sheet)

spark_store_df = spark.createDataFrame(store_df)
spark_store_df.write.mode("overwrite").saveAsTable("Rider_Base_Client")

print(f"Successfully saved {title} sheet into spark table Rider_Base_Client")
print("Saved table: Rider_Base_Client")

In [ ]:
# Adding a store mapping google sheet for riders
sheet = store_map.worksheet("Client 2 Data")

title, store_df = fetch_worksheet(sheet)

spark_store_df = spark.createDataFrame(store_df)
spark_store_df.write.mode("overwrite").saveAsTable("Rider_Stores_Client")

print(f"Successfully saved {title} sheet into spark table Rider_Stores_Client")
print("Saved table: Rider_Stores_Client")

In [ ]:
# Adding a store mapping google sheet for riders
sheet = rider_type.worksheet("Rider Type")

title, store_df = fetch_worksheet(sheet)

spark_store_df = spark.createDataFrame(store_df)
spark_store_df.write.mode("overwrite").saveAsTable("Rider_Type")

print(f"Successfully saved {title} sheet into spark table Rider_Type")
print("Saved table: Rider_Type")